In [14]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [31]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
        "output_config": {
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "task": {"type": "string"}
                    },
                    "additionalProperties": False
                }
            },
        }
    },
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [32]:
import json

prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""


In [ ]:

messages = []
add_user_message(messages, prompt)
#response = chat(messages, stop_sequences=["```json"])
response = chat(messages)

with open("005_test_dataset.json", "w") as f:
    json.dump(json.loads(response), f, indent=2)

[{'task': 'Write a Python function that validates an AWS IAM role ARN format and returns True if valid, False otherwise'}, {'task': 'Create a JSON object representing an AWS S3 bucket policy that allows public read access to all objects'}, {'task': 'Write a regular expression pattern that matches AWS EC2 instance IDs (format: i-followed by 17 hexadecimal characters)'}]
